In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [ ]:
X_train = pd.read_csv('./data/X_train.csv')
X_test = pd.read_csv('./data/X_test.csv')
y_train = pd.read_csv('./data/y_train.csv').values.ravel()
y_test = pd.read_csv('./data/y_test.csv').values.ravel()

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train class distribution:')
print(pd.Series(y_train).value_counts(normalize=True))
print('y_test class distribution:')
print(pd.Series(y_test).value_counts(normalize=True))

categorical_features = ['PetType', 'Breed', 'Color', 'Size']
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])


In [ ]:
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, solver='liblinear'))
])

baseline_pipeline.fit(X_train, y_train)
baseline_pred = baseline_pipeline.predict(X_test)
baseline_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

print('Wyniki modelu bazowego:')
print(classification_report(y_test, baseline_pred))
print('Accuracy:', accuracy_score(y_test, baseline_pred))
print('AUC-ROC:', roc_auc_score(y_test, baseline_proba))

cm = confusion_matrix(y_test, baseline_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
plt.title('Macierz pomyłek modelu bazowego')
plt.show()


## Badanie wpływu hiperparametrów


In [ ]:
C_values = np.logspace(-4, 4, 9)
train_scores = []
test_scores = []
roc_auc_scores = []

for C in C_values:
    model_C = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(C=C, penalty='l2', solver='liblinear', max_iter=1000, random_state=42))
    ])
    model_C.fit(X_train, y_train)
    train_scores.append(model_C.score(X_train, y_train))
    test_scores.append(model_C.score(X_test, y_test))
    y_proba = model_C.predict_proba(X_test)[:, 1]
    roc_auc_scores.append(roc_auc_score(y_test, y_proba))

plt.figure(figsize=(10, 5))
plt.semilogx(C_values, train_scores, marker='o', label='Dokładność trening')
plt.semilogx(C_values, test_scores, marker='o', label='Dokładność test')
plt.title('Wpływ parametru C na dokładność')
plt.xlabel('C (mniejsza wartość = silniejsza regularyzacja)')
plt.ylabel('Dokładność')
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.semilogx(C_values, roc_auc_scores, marker='o', color='tab:green')
plt.title('Wpływ parametru C na AUC-ROC')
plt.xlabel('C (mniejsza wartość = silniejsza regularyzacja)')
plt.ylabel('AUC-ROC')
plt.grid(True, which='both', linestyle='--', alpha=0.6)
plt.show()

penalties = ['l1', 'l2']
penalty_results = {'penalty': [], 'accuracy': [], 'roc_auc': []}

for penalty in penalties:
    model_penalty = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(C=1.0, penalty=penalty, solver='liblinear', max_iter=1000, random_state=42))
    ])
    model_penalty.fit(X_train, y_train)
    y_pred_pen = model_penalty.predict(X_test)
    y_proba_pen = model_penalty.predict_proba(X_test)[:, 1]
    penalty_results['penalty'].append(penalty)
    penalty_results['accuracy'].append(accuracy_score(y_test, y_pred_pen))
    penalty_results['roc_auc'].append(roc_auc_score(y_test, y_proba_pen))

plt.figure(figsize=(8, 4))
plt.bar(penalty_results['penalty'], penalty_results['accuracy'], color=['tab:blue', 'tab:orange'])
plt.title('Dokładność dla różnych rodzajów kary')
plt.xlabel('Penalty')
plt.ylabel('Dokładność')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(penalty_results['penalty'], penalty_results['roc_auc'], color=['tab:green', 'tab:red'])
plt.title('AUC-ROC dla różnych rodzajów kary')
plt.xlabel('Penalty')
plt.ylabel('AUC-ROC')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

print('Porównanie parametrów:')
for i, penalty in enumerate(penalty_results['penalty']):
    print(f"Penalty={penalty}: dokładność={penalty_results['accuracy'][i]:.4f}, auc-roc={penalty_results['roc_auc'][i]:.4f}")

param_grid = {
    'classifier__C': C_values,
    'classifier__penalty': ['l1', 'l2']
}
search = GridSearchCV(baseline_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
search.fit(X_train, y_train)

print('Najlepsze hiperparametry z GridSearchCV:')
print(search.best_params_)
print('Najlepszy wynik AUC-ROC w CV:', search.best_score_)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=search.best_params_['classifier__C'],
        penalty=search.best_params_['classifier__penalty'],
        solver='liblinear',
        max_iter=1000,
        random_state=42
    ))
])
final_pipeline.fit(X_train, y_train)
final_pred = final_pipeline.predict(X_test)
final_proba = final_pipeline.predict_proba(X_test)[:, 1]

print('Wyniki modelu końcowego:')
print(classification_report(y_test, final_pred))
print('Accuracy:', accuracy_score(y_test, final_pred))
print('AUC-ROC:', roc_auc_score(y_test, final_proba))
